# Week 20 Learning Notebook

This notebook mirrors the daily lesson plan and includes runnable demo code cells.

In [1]:
from datetime import date
import warnings
import numpy as np
import pandas as pd

try:
    import yfinance as yf
except Exception:
    yf = None

try:
    from pandas_datareader import data as pdr
except Exception:
    pdr = None

np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = '{:.6f}'.format

def _synthetic_price_frame(tickers, start='2018-01-01', periods=900, seed=7):
    idx = pd.bdate_range(start, periods=periods)
    out = pd.DataFrame(index=idx)
    rng = np.random.default_rng(seed)
    for i, ticker in enumerate(tickers):
        drift = 0.0002 + 0.00005 * (i + 1)
        vol = 0.010 + 0.002 * i
        rets = rng.normal(drift, vol, size=len(idx))
        out[ticker] = 100.0 * np.exp(np.cumsum(rets))
    return out

def load_market_prices(tickers, start='2018-01-01', end=None):
    end = end or date.today().isoformat()
    tickers = list(tickers)
    if yf is None:
        warnings.warn('yfinance unavailable, using synthetic data')
        return _synthetic_price_frame(tickers, start=start)

    try:
        raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)
        if isinstance(raw.columns, pd.MultiIndex):
            if 'Close' in raw.columns.get_level_values(0):
                close = raw['Close']
            else:
                close = raw.xs('Close', axis=1, level=0, drop_level=True)
        else:
            close = raw.rename(columns={raw.columns[0]: tickers[0]})
        close = close.reindex(columns=tickers)
        close = close.dropna(how='all').ffill().dropna()
        if close.empty:
            raise ValueError('empty market data from Yahoo Finance')
        return close
    except Exception as exc:
        warnings.warn(f'Yahoo download failed ({exc}); using synthetic data')
        return _synthetic_price_frame(tickers, start=start)

def load_fred_series(series_id, start='2018-01-01', end=None):
    end = end or date.today().isoformat()
    if pdr is None:
        warnings.warn('pandas_datareader unavailable, using synthetic macro data')
        idx = pd.bdate_range(start, end)
        vals = np.linspace(1.0, 1.2, len(idx)) + np.sin(np.arange(len(idx)) / 25) * 0.02
        return pd.Series(vals, index=idx, name=series_id)

    try:
        ser = pdr.DataReader(series_id, 'fred', start, end)[series_id]
        ser = ser.ffill().dropna()
        if ser.empty:
            raise ValueError('empty FRED series')
        return ser
    except Exception as exc:
        warnings.warn(f'FRED download failed ({exc}); using synthetic macro data')
        idx = pd.bdate_range(start, end)
        vals = np.linspace(1.0, 1.2, len(idx)) + np.sin(np.arange(len(idx)) / 25) * 0.02
        return pd.Series(vals, index=idx, name=series_id)

# Week 20 Day 01: Market microstructure essentials

## Study Duration
- Planned effort: 6-10 hours/day
- Required minimum: 6 hours (core + required extension); optional deep work extends to 10 hours.

## 6-10 Hour Daily Structure
- **Core Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Core Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Core Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Core Block 4 (60 min):** Python/pandas implementation and output verification.
- **Core Block 5 (30 min):** Practice questions, interview drill, and reflection.
- **Required Extension Block A (60 min):** Re-run the real trading example with one alternate ticker and one stress window.
- **Required Extension Block B (60 min):** Restart kernel and rerun all coding cells end-to-end, then add one extra validation test.
- **Optional Deep Work (0-4 hours):** Expand diagnostics, add edge-case tests, and improve interview-ready explanations.

## Why It Matters in Quant
Incorporate execution realism and microstructure awareness into a blended strategy portfolio.

## Continuity and Handoff
- Previous checkpoint: Week 19 Day 07: Portfolio Project
- Previous lesson file: content/week-19/day-07.md
- Today's deliverable: Create synthetic microstructure scenarios and expected slippage estimates.
- Next handoff target: Week 20 Day 02: Execution cost modeling
- Next lesson file: content/week-20/day-02.md

## Theory Concepts

### Concept 1: Order book dynamics
Order book dynamics is a core part of 'Execution, microstructure, and multi-strategy blend'. Start with notation discipline: define universe construction, signal scaling, and execution units before evaluating alpha. Then focus on alpha stability, execution realism, and risk-governed deployment by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Spread and depth
Spread and depth is a core part of 'Execution, microstructure, and multi-strategy blend'. Start with notation discipline: define universe construction, signal scaling, and execution units before evaluating alpha. Then focus on alpha stability, execution realism, and risk-governed deployment by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Latency and queue effects
Latency and queue effects is a core part of 'Execution, microstructure, and multi-strategy blend'. Start with notation discipline: define universe construction, signal scaling, and execution units before evaluating alpha. Then focus on alpha stability, execution realism, and risk-governed deployment by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Cross-Sectional Z
$$
z_{i,t}=\frac{x_{i,t}-\mu_t}{\sigma_t}
$$
**Plain-English interpretation:** Universe-normalized signal.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Measure pre-cost and post-cost values to verify execution frictions do not erase signal edge.

### Formula 2: Information Coefficient
$$
IC_t=Corr(score_{i,t},r_{i,t+1})
$$
**Plain-English interpretation:** Signal/forward-return linkage.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Measure pre-cost and post-cost values to verify execution frictions do not erase signal edge.

### Formula 3: IC t-Statistic
$$
t_{IC}=\frac{\bar{IC}}{Std(IC)/\sqrt{T}}
$$
**Plain-English interpretation:** Signal persistence test.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Measure pre-cost and post-cost values to verify execution frictions do not erase signal edge.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $IC$ | Information coefficient | correlation | 0.04 |
| $ADV$ | Average daily volume | shares/day | 12M |
| $IS$ | Implementation shortfall | basis points | 14.2 bps |

## Real Trading Example
- Instruments: SPY, IWM, EFA, EEM
- Macro overlay (FRED): DFF, BAMLH0A0HYM2
- Suggested window: 2018-01-01 to 2026-03-31
- Stress windows to inspect: 2020-03 to 2020-05, 2023-03 to 2023-06
- Scenario context: crowded-factor unwind and liquidity stress
- Day objective: Discuss how spread/depth changes alter expected execution quality.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull listed FRED series and join strictly by release-aware timestamps (no look-ahead).
3. Compute today's formulas and compare calm vs stress-window behavior.
4. Translate outputs into one explicit trade action and one hard risk guardrail.
5. Validate that the decision is consistent with topic 'Market microstructure essentials'.

## Step-by-Step Solved Problems
### Solved Problem 1: Factor z-score
Given:
- Signal=1.60, mean=0.70, std=0.45.
Solution:
1. $z=\frac{x-\mu}{\sigma}$.
2. z=(1.60-0.70)/0.45 = 2.00.
Final answer: Signal z-score = 2.00.
Common trap: Reporting gross signal performance without implementation costs or capacity constraints.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: IC t-stat
Given:
- Mean IC=0.045, std(IC)=0.018, T=12 months.
Solution:
1. $t=\frac{\bar{IC}}{s/\sqrt{T}}$.
2. t = 0.045/(0.018/sqrt(12)) = 8.66.
Final answer: IC t-stat = 8.66.
Common trap: Reporting gross signal performance without implementation costs or capacity constraints.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Implementation shortfall
Given:
- Arrival=101.20, execution=101.36.
Solution:
1. $IS_{bps}=10^4\frac{p_{exec}-p_{arr}}{p_{arr}}$.
2. IS_bps = 10000*(0.16/101.20) = 15.81.
Final answer: Implementation shortfall = 15.81 bps.
Common trap: Reporting gross signal performance without implementation costs or capacity constraints.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Create synthetic microstructure scenarios and expected slippage estimates.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
factor_scores = compute_factor_scores(universe_frame)
signals = build_long_short_buckets(factor_scores, q=0.2)
execution_cost = estimate_slippage(signals, adv_frame)
net_pnl = backtest_with_costs(signals, returns, execution_cost)
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 20 Day 01): Explain Cross-Sectional Z and define every symbol clearly for a crowded-factor unwind with liquidity deterioration.
   - Model answer: "I use Cross-Sectional Z as a decision bridge from market observations to position sizing. The formula is $z_{i,t}=\frac{x_{i,t}-\mu_t}{\sigma_t}$. I define each symbol with units first, then compute one concrete value, and finally state what trade action changes because of the result in this regime."
2. Risk manager question: Using one real ticker from this lesson, what hard guardrail would you enforce before live deployment?
   - Model answer: "I would run the workflow on SPY and a stress-sensitive peer, then throttle position sizing when estimated implementation shortfall rises above threshold. If the guardrail triggers, I switch to paper-trade monitoring and block new risk until diagnostics re-pass."
3. Data integrity question: Which checks must pass before you trust the output and place risk?
   - Model answer: "Before trading I verify survivorship-bias controls, borrow/shortability flags, and capacity-aware data completeness checks. If any check fails, I classify the run as non-tradable and log the incident."
4. Production question: Why does 'Market microstructure essentials' matter in live trading systems?
   - Model answer: "Market microstructure essentials matters because signal quality is meaningless unless net-of-cost and capacity-aware in live execution. In production I need reproducible calculations, explicit control limits, and escalation rules that survive stress windows."
5. Decision question: If your key metric degrades for three consecutive sessions, what is your fallback plan?
   - Model answer: "I halve position sizes, widen execution limits, and re-open capacity only after slippage improves. I only restore risk after rerun evidence confirms that assumptions are stable again."

Scoring rubric:
- 10/10: correct notation, one numeric example, explicit guardrail, data checks, and escalation path.
- 8/10: mostly correct notation plus a clear guardrail and fallback action.
- 6/10: partial correctness but vague controls or missing data validation.
- Below 6/10: formula recall without decision-quality risk controls.

### Interview Drill
- Prompt: "Walk me through Market microstructure essentials in an execution review with rising slippage and crowding risk."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision under constraints.
  3. Awareness of edge cases, costs, and failure modes.
  4. Clear escalation rule when guardrails are breached.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula, assumptions, and validation checks clearly.
  - Decision: explain one actionable rule, one risk guardrail, and one fallback action.

## Required Extension Track (2+ Hours)
- Re-run today's notebook from a fresh kernel so outputs are reproducible without hidden state.
- Add one additional risk guardrail and verify how it changes trade/no-trade decisions.
- Document one failure mode, one mitigation, and one escalation rule for production use.

Extension completion checks:
- [ ] Notebook restarted and all coding cells rerun successfully
- [ ] At least one extra validation/edge-case test added
- [ ] Risk guardrail and fallback action documented

## Reflection Question
Which microstructure feature most impacts your strategy style?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Full notebook rerun completed from clean kernel
- [ ] Reflection logged in progress tracker


## Week 20 Day 01 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [2]:
# Advanced strategy demo: cross-sectional momentum spread on real assets
np.random.seed(2001)
assets = ['SPY', 'QQQ', 'IWM', 'EFA', 'EEM', 'TLT', 'GLD', 'HYG']
prices = load_market_prices(assets, start='2015-01-01')

mom_63 = prices.pct_change(63).iloc[-1]
next_5 = prices.pct_change().tail(5).mean()
signals = mom_63.dropna().sort_values(ascending=False)

n = max(2, len(signals) // 4)
long_bucket = list(signals.head(n).index)
short_bucket = list(signals.tail(n).index)
spread = float(next_5.loc[long_bucket].mean() - next_5.loc[short_bucket].mean())

{
    'long_bucket': long_bucket,
    'short_bucket': short_bucket,
    'spread_return_proxy': spread,
    'signal_snapshot': signals.round(4).to_dict(),
}


{'long_bucket': ['EEM', 'GLD'],
 'short_bucket': ['HYG', 'TLT'],
 'spread_return_proxy': 0.005495435804020488,
 'signal_snapshot': {'EEM': 0.0955,
  'GLD': 0.0534,
  'EFA': 0.051,
  'QQQ': 0.0449,
  'IWM': 0.0405,
  'SPY': 0.0287,
  'HYG': 0.0099,
  'TLT': -0.003}}

## ReAct Verification Cell
Validate trade logic with a risk guardrail before reading the model quiz answers.

In [3]:
# ReAct-style verification: observe -> reason -> act -> verify
np.random.seed(11001)

observe_tickers = ['SPY', 'QQQ', 'TLT']
observe_prices = load_market_prices(observe_tickers, start='2020-01-01')
observe_returns = observe_prices.pct_change().dropna()

if observe_returns.empty:
    raise ValueError('No returns available for verification checks')

ann_vol = float(observe_returns['SPY'].std() * np.sqrt(252))
ann_ret = float((1 + observe_returns['SPY']).prod() ** (252 / len(observe_returns)) - 1)
sharpe_proxy = float((ann_ret - 0.03) / max(ann_vol, 1e-8))

# Risk-first deployment gate used in realistic interview responses.
guardrail = 'de-risk' if ann_vol > 0.30 else 'monitor'
decision = 'deploy-paper-trade' if sharpe_proxy > 0.40 and guardrail == 'monitor' else 'hold-and-review'

verification = {
    'topic': 'Market microstructure essentials',
    'week': 20,
    'day': 1,
    'observe_annual_return': ann_ret,
    'observe_annual_vol': ann_vol,
    'reason_sharpe_proxy': sharpe_proxy,
    'act_guardrail': guardrail,
    'verify_decision': decision,
}

verification


{'topic': 'Market microstructure essentials',
 'week': 20,
 'day': 1,
 'observe_annual_return': 0.14923384092579295,
 'observe_annual_vol': 0.20501276581162983,
 'reason_sharpe_proxy': 0.5815922752603006,
 'act_guardrail': 'monitor',
 'verify_decision': 'deploy-paper-trade'}

## Week 20 Day 01 Quiz

Topic: **Market microstructure essentials**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Explain Cross-Sectional Z-Score (z = (x - mu) / sigma) and define every symbol clearly.
2. Risk question: What hard guardrail would you enforce before deployment?
3. Data question: Which validation checks must pass before trusting this output?
4. Production question: Why does this topic matter in live systems, not just in notebooks?
5. Escalation question: If the metric degrades for three sessions, what is your fallback action?

Hint for Q2: throttle sizing when implementation shortfall worsens beyond policy.
Hint for Q3: capacity and borrow constraints before execution
Scoring: full credit requires notation correctness, one numeric example, explicit guardrail, and fallback path.

In [4]:
# Quiz model answers (auto-generated)
signal_x = 1.48
signal_mu = 0.92
signal_sigma = 0.28
metric_value = (signal_x - signal_mu) / signal_sigma
metric_name = 'cross_sectional_z'

print('Interview Question 1 (model answer):')
print('  Formula focus: Cross-Sectional Z-Score')
print('  Formula text : z = (x - mu) / sigma')
print('  Scenario     : crowded-factor unwind')
print('  Numeric value:', round(float(metric_value), 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce this guardrail before deployment:')
print('  throttle sizing when implementation shortfall exceeds threshold.')

print('\nInterview Question 3 (model answer):')
print('  I validate timestamps, missing data handling, and assumption consistency before taking risk.')

print('\nInterview Question 4 (model answer):')
print('  Topic:', 'Market microstructure essentials')
print('  This matters because production systems need reproducible metrics, explicit controls,')
print('  and a fallback decision path when stress conditions invalidate baseline assumptions.')

print('\nInterview Question 5 (model answer):')
print('  Fallback action: halve sizing and tighten execution limits.')

print('\nNumeric verification:')
print('  metric_name  =', metric_name)
print('  metric_value =', round(float(metric_value), 6))


Interview Question 1 (model answer):
  Formula focus: Cross-Sectional Z-Score
  Formula text : z = (x - mu) / sigma
  Scenario     : crowded-factor unwind
  Numeric value: 2.0

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce this guardrail before deployment:
  throttle sizing when implementation shortfall exceeds threshold.

Interview Question 3 (model answer):
  I validate timestamps, missing data handling, and assumption consistency before taking risk.

Interview Question 4 (model answer):
  Topic: Market microstructure essentials
  This matters because production systems need reproducible metrics, explicit controls,
  and a fallback decision path when stress conditions invalidate baseline assumptions.

Interview Question 5 (model answer):
  Fallback action: halve sizing and tighten execution limits.

Numeric verification:
  metric_name  = cross_sectional_z
  metric_value = 2.0


# Week 20 Day 02: Execution cost modeling

## Study Duration
- Planned effort: 6-10 hours/day
- Required minimum: 6 hours (core + required extension); optional deep work extends to 10 hours.

## 6-10 Hour Daily Structure
- **Core Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Core Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Core Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Core Block 4 (60 min):** Python/pandas implementation and output verification.
- **Core Block 5 (30 min):** Practice questions, interview drill, and reflection.
- **Required Extension Block A (60 min):** Re-run the real trading example with one alternate ticker and one stress window.
- **Required Extension Block B (60 min):** Restart kernel and rerun all coding cells end-to-end, then add one extra validation test.
- **Optional Deep Work (0-4 hours):** Expand diagnostics, add edge-case tests, and improve interview-ready explanations.

## Why It Matters in Quant
Incorporate execution realism and microstructure awareness into a blended strategy portfolio.

## Continuity and Handoff
- Previous checkpoint: Week 20 Day 01: Market microstructure essentials
- Previous lesson file: content/week-20/day-01.md
- Today's deliverable: Build cost model including spread and impact assumptions.
- Next handoff target: Week 20 Day 03: Strategy blending and diversification
- Next lesson file: content/week-20/day-03.md

## Theory Concepts

### Concept 1: Implementation shortfall
Implementation shortfall is a core part of 'Execution, microstructure, and multi-strategy blend'. Start with notation discipline: define universe construction, signal scaling, and execution units before evaluating alpha. Then focus on alpha stability, execution realism, and risk-governed deployment by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Impact models
Impact models is a core part of 'Execution, microstructure, and multi-strategy blend'. Start with notation discipline: define universe construction, signal scaling, and execution units before evaluating alpha. Then focus on alpha stability, execution realism, and risk-governed deployment by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Participation constraints
Participation constraints is a core part of 'Execution, microstructure, and multi-strategy blend'. Start with notation discipline: define universe construction, signal scaling, and execution units before evaluating alpha. Then focus on alpha stability, execution realism, and risk-governed deployment by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Information Coefficient
$$
IC_t=Corr(score_{i,t},r_{i,t+1})
$$
**Plain-English interpretation:** Signal/forward-return linkage.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Measure pre-cost and post-cost values to verify execution frictions do not erase signal edge.

### Formula 2: IC t-Statistic
$$
t_{IC}=\frac{\bar{IC}}{Std(IC)/\sqrt{T}}
$$
**Plain-English interpretation:** Signal persistence test.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Measure pre-cost and post-cost values to verify execution frictions do not erase signal edge.

### Formula 3: Spread Z-Score
$$
z_t=\frac{s_t-\mu_s}{\sigma_s}
$$
**Plain-English interpretation:** Stat-arb entry normalization.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Measure pre-cost and post-cost values to verify execution frictions do not erase signal edge.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $IC$ | Information coefficient | correlation | 0.04 |
| $ADV$ | Average daily volume | shares/day | 12M |
| $IS$ | Implementation shortfall | basis points | 14.2 bps |

## Real Trading Example
- Instruments: SPY, IWM, EFA, EEM
- Macro overlay (FRED): DFF, BAMLH0A0HYM2
- Suggested window: 2018-01-01 to 2026-03-31
- Stress windows to inspect: 2020-03 to 2020-05, 2023-03 to 2023-06
- Scenario context: crowded-factor unwind and liquidity stress
- Day objective: Estimate implementation shortfall under different participation rates.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull listed FRED series and join strictly by release-aware timestamps (no look-ahead).
3. Compute today's formulas and compare calm vs stress-window behavior.
4. Translate outputs into one explicit trade action and one hard risk guardrail.
5. Validate that the decision is consistent with topic 'Execution cost modeling'.

## Step-by-Step Solved Problems
### Solved Problem 1: Factor z-score
Given:
- Signal=1.60, mean=0.70, std=0.45.
Solution:
1. $z=\frac{x-\mu}{\sigma}$.
2. z=(1.60-0.70)/0.45 = 2.00.
Final answer: Signal z-score = 2.00.
Common trap: Reporting gross signal performance without implementation costs or capacity constraints.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: IC t-stat
Given:
- Mean IC=0.045, std(IC)=0.018, T=12 months.
Solution:
1. $t=\frac{\bar{IC}}{s/\sqrt{T}}$.
2. t = 0.045/(0.018/sqrt(12)) = 8.66.
Final answer: IC t-stat = 8.66.
Common trap: Reporting gross signal performance without implementation costs or capacity constraints.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Implementation shortfall
Given:
- Arrival=101.20, execution=101.36.
Solution:
1. $IS_{bps}=10^4\frac{p_{exec}-p_{arr}}{p_{arr}}$.
2. IS_bps = 10000*(0.16/101.20) = 15.81.
Final answer: Implementation shortfall = 15.81 bps.
Common trap: Reporting gross signal performance without implementation costs or capacity constraints.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Build cost model including spread and impact assumptions.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
factor_scores = compute_factor_scores(universe_frame)
signals = build_long_short_buckets(factor_scores, q=0.2)
execution_cost = estimate_slippage(signals, adv_frame)
net_pnl = backtest_with_costs(signals, returns, execution_cost)
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 20 Day 02): Explain Information Coefficient and define every symbol clearly for a crowded-factor unwind with liquidity deterioration.
   - Model answer: "I use Information Coefficient as a decision bridge from market observations to position sizing. The formula is $IC_t=Corr(score_{i,t},r_{i,t+1})$. I define each symbol with units first, then compute one concrete value, and finally state what trade action changes because of the result in this regime."
2. Risk manager question: Using one real ticker from this lesson, what hard guardrail would you enforce before live deployment?
   - Model answer: "I would run the workflow on SPY and a stress-sensitive peer, then throttle position sizing when estimated implementation shortfall rises above threshold. If the guardrail triggers, I switch to paper-trade monitoring and block new risk until diagnostics re-pass."
3. Data integrity question: Which checks must pass before you trust the output and place risk?
   - Model answer: "Before trading I verify survivorship-bias controls, borrow/shortability flags, and capacity-aware data completeness checks. If any check fails, I classify the run as non-tradable and log the incident."
4. Production question: Why does 'Execution cost modeling' matter in live trading systems?
   - Model answer: "Execution cost modeling matters because signal quality is meaningless unless net-of-cost and capacity-aware in live execution. In production I need reproducible calculations, explicit control limits, and escalation rules that survive stress windows."
5. Decision question: If your key metric degrades for three consecutive sessions, what is your fallback plan?
   - Model answer: "I halve position sizes, widen execution limits, and re-open capacity only after slippage improves. I only restore risk after rerun evidence confirms that assumptions are stable again."

Scoring rubric:
- 10/10: correct notation, one numeric example, explicit guardrail, data checks, and escalation path.
- 8/10: mostly correct notation plus a clear guardrail and fallback action.
- 6/10: partial correctness but vague controls or missing data validation.
- Below 6/10: formula recall without decision-quality risk controls.

### Interview Drill
- Prompt: "Walk me through Execution cost modeling in an execution review with rising slippage and crowding risk."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision under constraints.
  3. Awareness of edge cases, costs, and failure modes.
  4. Clear escalation rule when guardrails are breached.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula, assumptions, and validation checks clearly.
  - Decision: explain one actionable rule, one risk guardrail, and one fallback action.

## Required Extension Track (2+ Hours)
- Re-run today's notebook from a fresh kernel so outputs are reproducible without hidden state.
- Add one additional risk guardrail and verify how it changes trade/no-trade decisions.
- Document one failure mode, one mitigation, and one escalation rule for production use.

Extension completion checks:
- [ ] Notebook restarted and all coding cells rerun successfully
- [ ] At least one extra validation/edge-case test added
- [ ] Risk guardrail and fallback action documented

## Reflection Question
What execution assumption is least defensible in your current model?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Full notebook rerun completed from clean kernel
- [ ] Reflection logged in progress tracker


## Week 20 Day 02 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [5]:
# Advanced strategy demo: cross-sectional momentum spread on real assets
np.random.seed(2002)
assets = ['SPY', 'QQQ', 'IWM', 'EFA', 'EEM', 'TLT', 'GLD', 'HYG']
prices = load_market_prices(assets, start='2015-01-01')

mom_63 = prices.pct_change(63).iloc[-1]
next_5 = prices.pct_change().tail(5).mean()
signals = mom_63.dropna().sort_values(ascending=False)

n = max(2, len(signals) // 4)
long_bucket = list(signals.head(n).index)
short_bucket = list(signals.tail(n).index)
spread = float(next_5.loc[long_bucket].mean() - next_5.loc[short_bucket].mean())

{
    'long_bucket': long_bucket,
    'short_bucket': short_bucket,
    'spread_return_proxy': spread,
    'signal_snapshot': signals.round(4).to_dict(),
}


{'long_bucket': ['EEM', 'GLD'],
 'short_bucket': ['HYG', 'TLT'],
 'spread_return_proxy': 0.005495435804020488,
 'signal_snapshot': {'EEM': 0.0955,
  'GLD': 0.0534,
  'EFA': 0.051,
  'QQQ': 0.0449,
  'IWM': 0.0405,
  'SPY': 0.0287,
  'HYG': 0.0099,
  'TLT': -0.003}}

## ReAct Verification Cell
Validate trade logic with a risk guardrail before reading the model quiz answers.

In [6]:
# ReAct-style verification: observe -> reason -> act -> verify
np.random.seed(11002)

observe_tickers = ['SPY', 'QQQ', 'TLT']
observe_prices = load_market_prices(observe_tickers, start='2020-01-01')
observe_returns = observe_prices.pct_change().dropna()

if observe_returns.empty:
    raise ValueError('No returns available for verification checks')

ann_vol = float(observe_returns['SPY'].std() * np.sqrt(252))
ann_ret = float((1 + observe_returns['SPY']).prod() ** (252 / len(observe_returns)) - 1)
sharpe_proxy = float((ann_ret - 0.03) / max(ann_vol, 1e-8))

# Risk-first deployment gate used in realistic interview responses.
guardrail = 'de-risk' if ann_vol > 0.30 else 'monitor'
decision = 'deploy-paper-trade' if sharpe_proxy > 0.40 and guardrail == 'monitor' else 'hold-and-review'

verification = {
    'topic': 'Execution cost modeling',
    'week': 20,
    'day': 2,
    'observe_annual_return': ann_ret,
    'observe_annual_vol': ann_vol,
    'reason_sharpe_proxy': sharpe_proxy,
    'act_guardrail': guardrail,
    'verify_decision': decision,
}

verification


{'topic': 'Execution cost modeling',
 'week': 20,
 'day': 2,
 'observe_annual_return': 0.14923384092579295,
 'observe_annual_vol': 0.20501276581162983,
 'reason_sharpe_proxy': 0.5815922752603006,
 'act_guardrail': 'monitor',
 'verify_decision': 'deploy-paper-trade'}

## Week 20 Day 02 Quiz

Topic: **Execution cost modeling**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Explain Cross-Sectional Z-Score (z = (x - mu) / sigma) and define every symbol clearly.
2. Risk question: What hard guardrail would you enforce before deployment?
3. Data question: Which validation checks must pass before trusting this output?
4. Production question: Why does this topic matter in live systems, not just in notebooks?
5. Escalation question: If the metric degrades for three sessions, what is your fallback action?

Hint for Q2: throttle sizing when implementation shortfall worsens beyond policy.
Hint for Q3: capacity and borrow constraints before execution
Scoring: full credit requires notation correctness, one numeric example, explicit guardrail, and fallback path.

In [7]:
# Quiz model answers (auto-generated)
signal_x = 1.48
signal_mu = 0.92
signal_sigma = 0.28
metric_value = (signal_x - signal_mu) / signal_sigma
metric_name = 'cross_sectional_z'

print('Interview Question 1 (model answer):')
print('  Formula focus: Cross-Sectional Z-Score')
print('  Formula text : z = (x - mu) / sigma')
print('  Scenario     : crowded-factor unwind')
print('  Numeric value:', round(float(metric_value), 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce this guardrail before deployment:')
print('  throttle sizing when implementation shortfall exceeds threshold.')

print('\nInterview Question 3 (model answer):')
print('  I validate timestamps, missing data handling, and assumption consistency before taking risk.')

print('\nInterview Question 4 (model answer):')
print('  Topic:', 'Execution cost modeling')
print('  This matters because production systems need reproducible metrics, explicit controls,')
print('  and a fallback decision path when stress conditions invalidate baseline assumptions.')

print('\nInterview Question 5 (model answer):')
print('  Fallback action: halve sizing and tighten execution limits.')

print('\nNumeric verification:')
print('  metric_name  =', metric_name)
print('  metric_value =', round(float(metric_value), 6))


Interview Question 1 (model answer):
  Formula focus: Cross-Sectional Z-Score
  Formula text : z = (x - mu) / sigma
  Scenario     : crowded-factor unwind
  Numeric value: 2.0

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce this guardrail before deployment:
  throttle sizing when implementation shortfall exceeds threshold.

Interview Question 3 (model answer):
  I validate timestamps, missing data handling, and assumption consistency before taking risk.

Interview Question 4 (model answer):
  Topic: Execution cost modeling
  This matters because production systems need reproducible metrics, explicit controls,
  and a fallback decision path when stress conditions invalidate baseline assumptions.

Interview Question 5 (model answer):
  Fallback action: halve sizing and tighten execution limits.

Numeric verification:
  metric_name  = cross_sectional_z
  metric_value = 2.0


# Week 20 Day 03: Strategy blending and diversification

## Study Duration
- Planned effort: 6-10 hours/day
- Required minimum: 6 hours (core + required extension); optional deep work extends to 10 hours.

## 6-10 Hour Daily Structure
- **Core Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Core Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Core Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Core Block 4 (60 min):** Python/pandas implementation and output verification.
- **Core Block 5 (30 min):** Practice questions, interview drill, and reflection.
- **Required Extension Block A (60 min):** Re-run the real trading example with one alternate ticker and one stress window.
- **Required Extension Block B (60 min):** Restart kernel and rerun all coding cells end-to-end, then add one extra validation test.
- **Optional Deep Work (0-4 hours):** Expand diagnostics, add edge-case tests, and improve interview-ready explanations.

## Why It Matters in Quant
Incorporate execution realism and microstructure awareness into a blended strategy portfolio.

## Continuity and Handoff
- Previous checkpoint: Week 20 Day 02: Execution cost modeling
- Previous lesson file: content/week-20/day-02.md
- Today's deliverable: Implement multi-sleeve blend with adjustable risk budgets.
- Next handoff target: Week 20 Day 04: Rebalancing and execution scheduling
- Next lesson file: content/week-20/day-04.md

## Theory Concepts

### Concept 1: Correlation across alpha sleeves
Correlation across alpha sleeves is a core part of 'Execution, microstructure, and multi-strategy blend'. Start with notation discipline: define universe construction, signal scaling, and execution units before evaluating alpha. Then focus on alpha stability, execution realism, and risk-governed deployment by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Risk budget allocation
Risk budget allocation is a core part of 'Execution, microstructure, and multi-strategy blend'. Start with notation discipline: define universe construction, signal scaling, and execution units before evaluating alpha. Then focus on alpha stability, execution realism, and risk-governed deployment by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Capacity constraints
Capacity constraints is a core part of 'Execution, microstructure, and multi-strategy blend'. Start with notation discipline: define universe construction, signal scaling, and execution units before evaluating alpha. Then focus on alpha stability, execution realism, and risk-governed deployment by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: IC t-Statistic
$$
t_{IC}=\frac{\bar{IC}}{Std(IC)/\sqrt{T}}
$$
**Plain-English interpretation:** Signal persistence test.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Measure pre-cost and post-cost values to verify execution frictions do not erase signal edge.

### Formula 2: Spread Z-Score
$$
z_t=\frac{s_t-\mu_s}{\sigma_s}
$$
**Plain-English interpretation:** Stat-arb entry normalization.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Measure pre-cost and post-cost values to verify execution frictions do not erase signal edge.

### Formula 3: Implementation Shortfall
$$
IS_{bps}=10^4\frac{p_{exec}-p_{arr}}{p_{arr}}
$$
**Plain-English interpretation:** Execution loss in bps.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Measure pre-cost and post-cost values to verify execution frictions do not erase signal edge.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $IC$ | Information coefficient | correlation | 0.04 |
| $ADV$ | Average daily volume | shares/day | 12M |
| $IS$ | Implementation shortfall | basis points | 14.2 bps |

## Real Trading Example
- Instruments: SPY, IWM, EFA, EEM
- Macro overlay (FRED): DFF, BAMLH0A0HYM2
- Suggested window: 2018-01-01 to 2026-03-31
- Stress windows to inspect: 2020-03 to 2020-05, 2023-03 to 2023-06
- Scenario context: crowded-factor unwind and liquidity stress
- Day objective: Blend momentum, mean-reversion, and factor sleeves under risk limits.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull listed FRED series and join strictly by release-aware timestamps (no look-ahead).
3. Compute today's formulas and compare calm vs stress-window behavior.
4. Translate outputs into one explicit trade action and one hard risk guardrail.
5. Validate that the decision is consistent with topic 'Strategy blending and diversification'.

## Step-by-Step Solved Problems
### Solved Problem 1: Factor z-score
Given:
- Signal=1.60, mean=0.70, std=0.45.
Solution:
1. $z=\frac{x-\mu}{\sigma}$.
2. z=(1.60-0.70)/0.45 = 2.00.
Final answer: Signal z-score = 2.00.
Common trap: Reporting gross signal performance without implementation costs or capacity constraints.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: IC t-stat
Given:
- Mean IC=0.045, std(IC)=0.018, T=12 months.
Solution:
1. $t=\frac{\bar{IC}}{s/\sqrt{T}}$.
2. t = 0.045/(0.018/sqrt(12)) = 8.66.
Final answer: IC t-stat = 8.66.
Common trap: Reporting gross signal performance without implementation costs or capacity constraints.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Implementation shortfall
Given:
- Arrival=101.20, execution=101.36.
Solution:
1. $IS_{bps}=10^4\frac{p_{exec}-p_{arr}}{p_{arr}}$.
2. IS_bps = 10000*(0.16/101.20) = 15.81.
Final answer: Implementation shortfall = 15.81 bps.
Common trap: Reporting gross signal performance without implementation costs or capacity constraints.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Implement multi-sleeve blend with adjustable risk budgets.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
factor_scores = compute_factor_scores(universe_frame)
signals = build_long_short_buckets(factor_scores, q=0.2)
execution_cost = estimate_slippage(signals, adv_frame)
net_pnl = backtest_with_costs(signals, returns, execution_cost)
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 20 Day 03): Explain IC t-Statistic and define every symbol clearly for a crowded-factor unwind with liquidity deterioration.
   - Model answer: "I use IC t-Statistic as a decision bridge from market observations to position sizing. The formula is $t_{IC}=\frac{\bar{IC}}{Std(IC)/\sqrt{T}}$. I define each symbol with units first, then compute one concrete value, and finally state what trade action changes because of the result in this regime."
2. Risk manager question: Using one real ticker from this lesson, what hard guardrail would you enforce before live deployment?
   - Model answer: "I would run the workflow on SPY and a stress-sensitive peer, then throttle position sizing when estimated implementation shortfall rises above threshold. If the guardrail triggers, I switch to paper-trade monitoring and block new risk until diagnostics re-pass."
3. Data integrity question: Which checks must pass before you trust the output and place risk?
   - Model answer: "Before trading I verify survivorship-bias controls, borrow/shortability flags, and capacity-aware data completeness checks. If any check fails, I classify the run as non-tradable and log the incident."
4. Production question: Why does 'Strategy blending and diversification' matter in live trading systems?
   - Model answer: "Strategy blending and diversification matters because signal quality is meaningless unless net-of-cost and capacity-aware in live execution. In production I need reproducible calculations, explicit control limits, and escalation rules that survive stress windows."
5. Decision question: If your key metric degrades for three consecutive sessions, what is your fallback plan?
   - Model answer: "I halve position sizes, widen execution limits, and re-open capacity only after slippage improves. I only restore risk after rerun evidence confirms that assumptions are stable again."

Scoring rubric:
- 10/10: correct notation, one numeric example, explicit guardrail, data checks, and escalation path.
- 8/10: mostly correct notation plus a clear guardrail and fallback action.
- 6/10: partial correctness but vague controls or missing data validation.
- Below 6/10: formula recall without decision-quality risk controls.

### Interview Drill
- Prompt: "Walk me through Strategy blending and diversification in an execution review with rising slippage and crowding risk."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision under constraints.
  3. Awareness of edge cases, costs, and failure modes.
  4. Clear escalation rule when guardrails are breached.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula, assumptions, and validation checks clearly.
  - Decision: explain one actionable rule, one risk guardrail, and one fallback action.

## Required Extension Track (2+ Hours)
- Re-run today's notebook from a fresh kernel so outputs are reproducible without hidden state.
- Add one additional risk guardrail and verify how it changes trade/no-trade decisions.
- Document one failure mode, one mitigation, and one escalation rule for production use.

Extension completion checks:
- [ ] Notebook restarted and all coding cells rerun successfully
- [ ] At least one extra validation/edge-case test added
- [ ] Risk guardrail and fallback action documented

## Reflection Question
How does capacity constraint change blend weights?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Full notebook rerun completed from clean kernel
- [ ] Reflection logged in progress tracker


## Week 20 Day 03 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [8]:
# Advanced strategy demo: cross-sectional momentum spread on real assets
np.random.seed(2003)
assets = ['SPY', 'QQQ', 'IWM', 'EFA', 'EEM', 'TLT', 'GLD', 'HYG']
prices = load_market_prices(assets, start='2015-01-01')

mom_63 = prices.pct_change(63).iloc[-1]
next_5 = prices.pct_change().tail(5).mean()
signals = mom_63.dropna().sort_values(ascending=False)

n = max(2, len(signals) // 4)
long_bucket = list(signals.head(n).index)
short_bucket = list(signals.tail(n).index)
spread = float(next_5.loc[long_bucket].mean() - next_5.loc[short_bucket].mean())

{
    'long_bucket': long_bucket,
    'short_bucket': short_bucket,
    'spread_return_proxy': spread,
    'signal_snapshot': signals.round(4).to_dict(),
}


{'long_bucket': ['EEM', 'GLD'],
 'short_bucket': ['HYG', 'TLT'],
 'spread_return_proxy': 0.005495435804020488,
 'signal_snapshot': {'EEM': 0.0955,
  'GLD': 0.0534,
  'EFA': 0.051,
  'QQQ': 0.0449,
  'IWM': 0.0405,
  'SPY': 0.0287,
  'HYG': 0.0099,
  'TLT': -0.003}}

## ReAct Verification Cell
Validate trade logic with a risk guardrail before reading the model quiz answers.

In [9]:
# ReAct-style verification: observe -> reason -> act -> verify
np.random.seed(11003)

observe_tickers = ['SPY', 'QQQ', 'TLT']
observe_prices = load_market_prices(observe_tickers, start='2020-01-01')
observe_returns = observe_prices.pct_change().dropna()

if observe_returns.empty:
    raise ValueError('No returns available for verification checks')

ann_vol = float(observe_returns['SPY'].std() * np.sqrt(252))
ann_ret = float((1 + observe_returns['SPY']).prod() ** (252 / len(observe_returns)) - 1)
sharpe_proxy = float((ann_ret - 0.03) / max(ann_vol, 1e-8))

# Risk-first deployment gate used in realistic interview responses.
guardrail = 'de-risk' if ann_vol > 0.30 else 'monitor'
decision = 'deploy-paper-trade' if sharpe_proxy > 0.40 and guardrail == 'monitor' else 'hold-and-review'

verification = {
    'topic': 'Strategy blending and diversification',
    'week': 20,
    'day': 3,
    'observe_annual_return': ann_ret,
    'observe_annual_vol': ann_vol,
    'reason_sharpe_proxy': sharpe_proxy,
    'act_guardrail': guardrail,
    'verify_decision': decision,
}

verification


{'topic': 'Strategy blending and diversification',
 'week': 20,
 'day': 3,
 'observe_annual_return': 0.14923384092579295,
 'observe_annual_vol': 0.20501276581162983,
 'reason_sharpe_proxy': 0.5815922752603006,
 'act_guardrail': 'monitor',
 'verify_decision': 'deploy-paper-trade'}

## Week 20 Day 03 Quiz

Topic: **Strategy blending and diversification**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Explain Cross-Sectional Z-Score (z = (x - mu) / sigma) and define every symbol clearly.
2. Risk question: What hard guardrail would you enforce before deployment?
3. Data question: Which validation checks must pass before trusting this output?
4. Production question: Why does this topic matter in live systems, not just in notebooks?
5. Escalation question: If the metric degrades for three sessions, what is your fallback action?

Hint for Q2: throttle sizing when implementation shortfall worsens beyond policy.
Hint for Q3: capacity and borrow constraints before execution
Scoring: full credit requires notation correctness, one numeric example, explicit guardrail, and fallback path.

In [10]:
# Quiz model answers (auto-generated)
signal_x = 1.48
signal_mu = 0.92
signal_sigma = 0.28
metric_value = (signal_x - signal_mu) / signal_sigma
metric_name = 'cross_sectional_z'

print('Interview Question 1 (model answer):')
print('  Formula focus: Cross-Sectional Z-Score')
print('  Formula text : z = (x - mu) / sigma')
print('  Scenario     : crowded-factor unwind')
print('  Numeric value:', round(float(metric_value), 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce this guardrail before deployment:')
print('  throttle sizing when implementation shortfall exceeds threshold.')

print('\nInterview Question 3 (model answer):')
print('  I validate timestamps, missing data handling, and assumption consistency before taking risk.')

print('\nInterview Question 4 (model answer):')
print('  Topic:', 'Strategy blending and diversification')
print('  This matters because production systems need reproducible metrics, explicit controls,')
print('  and a fallback decision path when stress conditions invalidate baseline assumptions.')

print('\nInterview Question 5 (model answer):')
print('  Fallback action: halve sizing and tighten execution limits.')

print('\nNumeric verification:')
print('  metric_name  =', metric_name)
print('  metric_value =', round(float(metric_value), 6))


Interview Question 1 (model answer):
  Formula focus: Cross-Sectional Z-Score
  Formula text : z = (x - mu) / sigma
  Scenario     : crowded-factor unwind
  Numeric value: 2.0

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce this guardrail before deployment:
  throttle sizing when implementation shortfall exceeds threshold.

Interview Question 3 (model answer):
  I validate timestamps, missing data handling, and assumption consistency before taking risk.

Interview Question 4 (model answer):
  Topic: Strategy blending and diversification
  This matters because production systems need reproducible metrics, explicit controls,
  and a fallback decision path when stress conditions invalidate baseline assumptions.

Interview Question 5 (model answer):
  Fallback action: halve sizing and tighten execution limits.

Numeric verification:
  metric_name  = cross_sectional_z
  metric_value = 2.0


# Week 20 Day 04: Rebalancing and execution scheduling

## Study Duration
- Planned effort: 6-10 hours/day
- Required minimum: 6 hours (core + required extension); optional deep work extends to 10 hours.

## 6-10 Hour Daily Structure
- **Core Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Core Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Core Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Core Block 4 (60 min):** Python/pandas implementation and output verification.
- **Core Block 5 (30 min):** Practice questions, interview drill, and reflection.
- **Required Extension Block A (60 min):** Re-run the real trading example with one alternate ticker and one stress window.
- **Required Extension Block B (60 min):** Restart kernel and rerun all coding cells end-to-end, then add one extra validation test.
- **Optional Deep Work (0-4 hours):** Expand diagnostics, add edge-case tests, and improve interview-ready explanations.

## Why It Matters in Quant
Incorporate execution realism and microstructure awareness into a blended strategy portfolio.

## Continuity and Handoff
- Previous checkpoint: Week 20 Day 03: Strategy blending and diversification
- Previous lesson file: content/week-20/day-03.md
- Today's deliverable: Create scheduler simulation for rebalance events.
- Next handoff target: Week 20 Day 05: Capstone assembly and governance
- Next lesson file: content/week-20/day-05.md

## Theory Concepts

### Concept 1: Trade scheduling
Trade scheduling is a core part of 'Execution, microstructure, and multi-strategy blend'. Start with notation discipline: define universe construction, signal scaling, and execution units before evaluating alpha. Then focus on alpha stability, execution realism, and risk-governed deployment by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Urgency vs cost tradeoff
Urgency vs cost tradeoff is a core part of 'Execution, microstructure, and multi-strategy blend'. Start with notation discipline: define universe construction, signal scaling, and execution units before evaluating alpha. Then focus on alpha stability, execution realism, and risk-governed deployment by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Execution windows
Execution windows is a core part of 'Execution, microstructure, and multi-strategy blend'. Start with notation discipline: define universe construction, signal scaling, and execution units before evaluating alpha. Then focus on alpha stability, execution realism, and risk-governed deployment by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Spread Z-Score
$$
z_t=\frac{s_t-\mu_s}{\sigma_s}
$$
**Plain-English interpretation:** Stat-arb entry normalization.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Measure pre-cost and post-cost values to verify execution frictions do not erase signal edge.

### Formula 2: Implementation Shortfall
$$
IS_{bps}=10^4\frac{p_{exec}-p_{arr}}{p_{arr}}
$$
**Plain-English interpretation:** Execution loss in bps.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Measure pre-cost and post-cost values to verify execution frictions do not erase signal edge.

### Formula 3: Cross-Sectional Z
$$
z_{i,t}=\frac{x_{i,t}-\mu_t}{\sigma_t}
$$
**Plain-English interpretation:** Universe-normalized signal.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Measure pre-cost and post-cost values to verify execution frictions do not erase signal edge.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $IC$ | Information coefficient | correlation | 0.04 |
| $ADV$ | Average daily volume | shares/day | 12M |
| $IS$ | Implementation shortfall | basis points | 14.2 bps |

## Real Trading Example
- Instruments: SPY, IWM, EFA, EEM
- Macro overlay (FRED): DFF, BAMLH0A0HYM2
- Suggested window: 2018-01-01 to 2026-03-31
- Stress windows to inspect: 2020-03 to 2020-05, 2023-03 to 2023-06
- Scenario context: crowded-factor unwind and liquidity stress
- Day objective: Compare immediate vs scheduled execution impact assumptions.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull listed FRED series and join strictly by release-aware timestamps (no look-ahead).
3. Compute today's formulas and compare calm vs stress-window behavior.
4. Translate outputs into one explicit trade action and one hard risk guardrail.
5. Validate that the decision is consistent with topic 'Rebalancing and execution scheduling'.

## Step-by-Step Solved Problems
### Solved Problem 1: Factor z-score
Given:
- Signal=1.60, mean=0.70, std=0.45.
Solution:
1. $z=\frac{x-\mu}{\sigma}$.
2. z=(1.60-0.70)/0.45 = 2.00.
Final answer: Signal z-score = 2.00.
Common trap: Reporting gross signal performance without implementation costs or capacity constraints.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: IC t-stat
Given:
- Mean IC=0.045, std(IC)=0.018, T=12 months.
Solution:
1. $t=\frac{\bar{IC}}{s/\sqrt{T}}$.
2. t = 0.045/(0.018/sqrt(12)) = 8.66.
Final answer: IC t-stat = 8.66.
Common trap: Reporting gross signal performance without implementation costs or capacity constraints.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Implementation shortfall
Given:
- Arrival=101.20, execution=101.36.
Solution:
1. $IS_{bps}=10^4\frac{p_{exec}-p_{arr}}{p_{arr}}$.
2. IS_bps = 10000*(0.16/101.20) = 15.81.
Final answer: Implementation shortfall = 15.81 bps.
Common trap: Reporting gross signal performance without implementation costs or capacity constraints.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Create scheduler simulation for rebalance events.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
factor_scores = compute_factor_scores(universe_frame)
signals = build_long_short_buckets(factor_scores, q=0.2)
execution_cost = estimate_slippage(signals, adv_frame)
net_pnl = backtest_with_costs(signals, returns, execution_cost)
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 20 Day 04): Explain Spread Z-Score and define every symbol clearly for a crowded-factor unwind with liquidity deterioration.
   - Model answer: "I use Spread Z-Score as a decision bridge from market observations to position sizing. The formula is $z_t=\frac{s_t-\mu_s}{\sigma_s}$. I define each symbol with units first, then compute one concrete value, and finally state what trade action changes because of the result in this regime."
2. Risk manager question: Using one real ticker from this lesson, what hard guardrail would you enforce before live deployment?
   - Model answer: "I would run the workflow on SPY and a stress-sensitive peer, then throttle position sizing when estimated implementation shortfall rises above threshold. If the guardrail triggers, I switch to paper-trade monitoring and block new risk until diagnostics re-pass."
3. Data integrity question: Which checks must pass before you trust the output and place risk?
   - Model answer: "Before trading I verify survivorship-bias controls, borrow/shortability flags, and capacity-aware data completeness checks. If any check fails, I classify the run as non-tradable and log the incident."
4. Production question: Why does 'Rebalancing and execution scheduling' matter in live trading systems?
   - Model answer: "Rebalancing and execution scheduling matters because signal quality is meaningless unless net-of-cost and capacity-aware in live execution. In production I need reproducible calculations, explicit control limits, and escalation rules that survive stress windows."
5. Decision question: If your key metric degrades for three consecutive sessions, what is your fallback plan?
   - Model answer: "I halve position sizes, widen execution limits, and re-open capacity only after slippage improves. I only restore risk after rerun evidence confirms that assumptions are stable again."

Scoring rubric:
- 10/10: correct notation, one numeric example, explicit guardrail, data checks, and escalation path.
- 8/10: mostly correct notation plus a clear guardrail and fallback action.
- 6/10: partial correctness but vague controls or missing data validation.
- Below 6/10: formula recall without decision-quality risk controls.

### Interview Drill
- Prompt: "Walk me through Rebalancing and execution scheduling in an execution review with rising slippage and crowding risk."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision under constraints.
  3. Awareness of edge cases, costs, and failure modes.
  4. Clear escalation rule when guardrails are breached.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula, assumptions, and validation checks clearly.
  - Decision: explain one actionable rule, one risk guardrail, and one fallback action.

## Required Extension Track (2+ Hours)
- Re-run today's notebook from a fresh kernel so outputs are reproducible without hidden state.
- Add one additional risk guardrail and verify how it changes trade/no-trade decisions.
- Document one failure mode, one mitigation, and one escalation rule for production use.

Extension completion checks:
- [ ] Notebook restarted and all coding cells rerun successfully
- [ ] At least one extra validation/edge-case test added
- [ ] Risk guardrail and fallback action documented

## Reflection Question
When should urgency override cost minimization?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Full notebook rerun completed from clean kernel
- [ ] Reflection logged in progress tracker


## Week 20 Day 04 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [11]:
# Advanced strategy demo: cross-sectional momentum spread on real assets
np.random.seed(2004)
assets = ['SPY', 'QQQ', 'IWM', 'EFA', 'EEM', 'TLT', 'GLD', 'HYG']
prices = load_market_prices(assets, start='2015-01-01')

mom_63 = prices.pct_change(63).iloc[-1]
next_5 = prices.pct_change().tail(5).mean()
signals = mom_63.dropna().sort_values(ascending=False)

n = max(2, len(signals) // 4)
long_bucket = list(signals.head(n).index)
short_bucket = list(signals.tail(n).index)
spread = float(next_5.loc[long_bucket].mean() - next_5.loc[short_bucket].mean())

{
    'long_bucket': long_bucket,
    'short_bucket': short_bucket,
    'spread_return_proxy': spread,
    'signal_snapshot': signals.round(4).to_dict(),
}


{'long_bucket': ['EEM', 'GLD'],
 'short_bucket': ['HYG', 'TLT'],
 'spread_return_proxy': 0.005495435804020488,
 'signal_snapshot': {'EEM': 0.0955,
  'GLD': 0.0534,
  'EFA': 0.051,
  'QQQ': 0.0449,
  'IWM': 0.0405,
  'SPY': 0.0287,
  'HYG': 0.0099,
  'TLT': -0.003}}

## ReAct Verification Cell
Validate trade logic with a risk guardrail before reading the model quiz answers.

In [12]:
# ReAct-style verification: observe -> reason -> act -> verify
np.random.seed(11004)

observe_tickers = ['SPY', 'QQQ', 'TLT']
observe_prices = load_market_prices(observe_tickers, start='2020-01-01')
observe_returns = observe_prices.pct_change().dropna()

if observe_returns.empty:
    raise ValueError('No returns available for verification checks')

ann_vol = float(observe_returns['SPY'].std() * np.sqrt(252))
ann_ret = float((1 + observe_returns['SPY']).prod() ** (252 / len(observe_returns)) - 1)
sharpe_proxy = float((ann_ret - 0.03) / max(ann_vol, 1e-8))

# Risk-first deployment gate used in realistic interview responses.
guardrail = 'de-risk' if ann_vol > 0.30 else 'monitor'
decision = 'deploy-paper-trade' if sharpe_proxy > 0.40 and guardrail == 'monitor' else 'hold-and-review'

verification = {
    'topic': 'Rebalancing and execution scheduling',
    'week': 20,
    'day': 4,
    'observe_annual_return': ann_ret,
    'observe_annual_vol': ann_vol,
    'reason_sharpe_proxy': sharpe_proxy,
    'act_guardrail': guardrail,
    'verify_decision': decision,
}

verification


{'topic': 'Rebalancing and execution scheduling',
 'week': 20,
 'day': 4,
 'observe_annual_return': 0.14923384092579295,
 'observe_annual_vol': 0.20501276581162983,
 'reason_sharpe_proxy': 0.5815922752603006,
 'act_guardrail': 'monitor',
 'verify_decision': 'deploy-paper-trade'}

## Week 20 Day 04 Quiz

Topic: **Rebalancing and execution scheduling**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Explain Cross-Sectional Z-Score (z = (x - mu) / sigma) and define every symbol clearly.
2. Risk question: What hard guardrail would you enforce before deployment?
3. Data question: Which validation checks must pass before trusting this output?
4. Production question: Why does this topic matter in live systems, not just in notebooks?
5. Escalation question: If the metric degrades for three sessions, what is your fallback action?

Hint for Q2: throttle sizing when implementation shortfall worsens beyond policy.
Hint for Q3: capacity and borrow constraints before execution
Scoring: full credit requires notation correctness, one numeric example, explicit guardrail, and fallback path.

In [13]:
# Quiz model answers (auto-generated)
signal_x = 1.48
signal_mu = 0.92
signal_sigma = 0.28
metric_value = (signal_x - signal_mu) / signal_sigma
metric_name = 'cross_sectional_z'

print('Interview Question 1 (model answer):')
print('  Formula focus: Cross-Sectional Z-Score')
print('  Formula text : z = (x - mu) / sigma')
print('  Scenario     : crowded-factor unwind')
print('  Numeric value:', round(float(metric_value), 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce this guardrail before deployment:')
print('  throttle sizing when implementation shortfall exceeds threshold.')

print('\nInterview Question 3 (model answer):')
print('  I validate timestamps, missing data handling, and assumption consistency before taking risk.')

print('\nInterview Question 4 (model answer):')
print('  Topic:', 'Rebalancing and execution scheduling')
print('  This matters because production systems need reproducible metrics, explicit controls,')
print('  and a fallback decision path when stress conditions invalidate baseline assumptions.')

print('\nInterview Question 5 (model answer):')
print('  Fallback action: halve sizing and tighten execution limits.')

print('\nNumeric verification:')
print('  metric_name  =', metric_name)
print('  metric_value =', round(float(metric_value), 6))


Interview Question 1 (model answer):
  Formula focus: Cross-Sectional Z-Score
  Formula text : z = (x - mu) / sigma
  Scenario     : crowded-factor unwind
  Numeric value: 2.0

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce this guardrail before deployment:
  throttle sizing when implementation shortfall exceeds threshold.

Interview Question 3 (model answer):
  I validate timestamps, missing data handling, and assumption consistency before taking risk.

Interview Question 4 (model answer):
  Topic: Rebalancing and execution scheduling
  This matters because production systems need reproducible metrics, explicit controls,
  and a fallback decision path when stress conditions invalidate baseline assumptions.

Interview Question 5 (model answer):
  Fallback action: halve sizing and tighten execution limits.

Numeric verification:
  metric_name  = cross_sectional_z
  metric_value = 2.0


# Week 20 Day 05: Capstone assembly and governance

## Study Duration
- Planned effort: 6-10 hours/day
- Required minimum: 6 hours (core + required extension); optional deep work extends to 10 hours.

## 6-10 Hour Daily Structure
- **Core Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Core Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Core Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Core Block 4 (60 min):** Python/pandas implementation and output verification.
- **Core Block 5 (30 min):** Practice questions, interview drill, and reflection.
- **Required Extension Block A (60 min):** Re-run the real trading example with one alternate ticker and one stress window.
- **Required Extension Block B (60 min):** Restart kernel and rerun all coding cells end-to-end, then add one extra validation test.
- **Optional Deep Work (0-4 hours):** Expand diagnostics, add edge-case tests, and improve interview-ready explanations.

## Why It Matters in Quant
Incorporate execution realism and microstructure awareness into a blended strategy portfolio.

## Continuity and Handoff
- Previous checkpoint: Week 20 Day 04: Rebalancing and execution scheduling
- Previous lesson file: content/week-20/day-04.md
- Today's deliverable: Generate final blend dashboard for capstone review.
- Next handoff target: Week 20 Day 06: Revision Sprint
- Next lesson file: content/week-20/day-06.md

## Theory Concepts

### Concept 1: Blend-level performance attribution
Blend-level performance attribution is a core part of 'Execution, microstructure, and multi-strategy blend'. Start with notation discipline: define universe construction, signal scaling, and execution units before evaluating alpha. Then focus on alpha stability, execution realism, and risk-governed deployment by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Execution-risk reporting
Execution-risk reporting is a core part of 'Execution, microstructure, and multi-strategy blend'. Start with notation discipline: define universe construction, signal scaling, and execution units before evaluating alpha. Then focus on alpha stability, execution realism, and risk-governed deployment by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Operational controls
Operational controls is a core part of 'Execution, microstructure, and multi-strategy blend'. Start with notation discipline: define universe construction, signal scaling, and execution units before evaluating alpha. Then focus on alpha stability, execution realism, and risk-governed deployment by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Implementation Shortfall
$$
IS_{bps}=10^4\frac{p_{exec}-p_{arr}}{p_{arr}}
$$
**Plain-English interpretation:** Execution loss in bps.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Measure pre-cost and post-cost values to verify execution frictions do not erase signal edge.

### Formula 2: Cross-Sectional Z
$$
z_{i,t}=\frac{x_{i,t}-\mu_t}{\sigma_t}
$$
**Plain-English interpretation:** Universe-normalized signal.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Measure pre-cost and post-cost values to verify execution frictions do not erase signal edge.

### Formula 3: Information Coefficient
$$
IC_t=Corr(score_{i,t},r_{i,t+1})
$$
**Plain-English interpretation:** Signal/forward-return linkage.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Measure pre-cost and post-cost values to verify execution frictions do not erase signal edge.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $IC$ | Information coefficient | correlation | 0.04 |
| $ADV$ | Average daily volume | shares/day | 12M |
| $IS$ | Implementation shortfall | basis points | 14.2 bps |

## Real Trading Example
- Instruments: SPY, IWM, EFA, EEM
- Macro overlay (FRED): DFF, BAMLH0A0HYM2
- Suggested window: 2018-01-01 to 2026-03-31
- Stress windows to inspect: 2020-03 to 2020-05, 2023-03 to 2023-06
- Scenario context: crowded-factor unwind and liquidity stress
- Day objective: Compile multi-strategy report with net-of-cost metrics.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull listed FRED series and join strictly by release-aware timestamps (no look-ahead).
3. Compute today's formulas and compare calm vs stress-window behavior.
4. Translate outputs into one explicit trade action and one hard risk guardrail.
5. Validate that the decision is consistent with topic 'Capstone assembly and governance'.

## Step-by-Step Solved Problems
### Solved Problem 1: Factor z-score
Given:
- Signal=1.60, mean=0.70, std=0.45.
Solution:
1. $z=\frac{x-\mu}{\sigma}$.
2. z=(1.60-0.70)/0.45 = 2.00.
Final answer: Signal z-score = 2.00.
Common trap: Reporting gross signal performance without implementation costs or capacity constraints.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: IC t-stat
Given:
- Mean IC=0.045, std(IC)=0.018, T=12 months.
Solution:
1. $t=\frac{\bar{IC}}{s/\sqrt{T}}$.
2. t = 0.045/(0.018/sqrt(12)) = 8.66.
Final answer: IC t-stat = 8.66.
Common trap: Reporting gross signal performance without implementation costs or capacity constraints.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Implementation shortfall
Given:
- Arrival=101.20, execution=101.36.
Solution:
1. $IS_{bps}=10^4\frac{p_{exec}-p_{arr}}{p_{arr}}$.
2. IS_bps = 10000*(0.16/101.20) = 15.81.
Final answer: Implementation shortfall = 15.81 bps.
Common trap: Reporting gross signal performance without implementation costs or capacity constraints.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Generate final blend dashboard for capstone review.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
factor_scores = compute_factor_scores(universe_frame)
signals = build_long_short_buckets(factor_scores, q=0.2)
execution_cost = estimate_slippage(signals, adv_frame)
net_pnl = backtest_with_costs(signals, returns, execution_cost)
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 20 Day 05): Explain Implementation Shortfall and define every symbol clearly for a crowded-factor unwind with liquidity deterioration.
   - Model answer: "I use Implementation Shortfall as a decision bridge from market observations to position sizing. The formula is $IS_{bps}=10^4\frac{p_{exec}-p_{arr}}{p_{arr}}$. I define each symbol with units first, then compute one concrete value, and finally state what trade action changes because of the result in this regime."
2. Risk manager question: Using one real ticker from this lesson, what hard guardrail would you enforce before live deployment?
   - Model answer: "I would run the workflow on SPY and a stress-sensitive peer, then throttle position sizing when estimated implementation shortfall rises above threshold. If the guardrail triggers, I switch to paper-trade monitoring and block new risk until diagnostics re-pass."
3. Data integrity question: Which checks must pass before you trust the output and place risk?
   - Model answer: "Before trading I verify survivorship-bias controls, borrow/shortability flags, and capacity-aware data completeness checks. If any check fails, I classify the run as non-tradable and log the incident."
4. Production question: Why does 'Capstone assembly and governance' matter in live trading systems?
   - Model answer: "Capstone assembly and governance matters because signal quality is meaningless unless net-of-cost and capacity-aware in live execution. In production I need reproducible calculations, explicit control limits, and escalation rules that survive stress windows."
5. Decision question: If your key metric degrades for three consecutive sessions, what is your fallback plan?
   - Model answer: "I halve position sizes, widen execution limits, and re-open capacity only after slippage improves. I only restore risk after rerun evidence confirms that assumptions are stable again."

Scoring rubric:
- 10/10: correct notation, one numeric example, explicit guardrail, data checks, and escalation path.
- 8/10: mostly correct notation plus a clear guardrail and fallback action.
- 6/10: partial correctness but vague controls or missing data validation.
- Below 6/10: formula recall without decision-quality risk controls.

### Interview Drill
- Prompt: "Walk me through Capstone assembly and governance in an execution review with rising slippage and crowding risk."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision under constraints.
  3. Awareness of edge cases, costs, and failure modes.
  4. Clear escalation rule when guardrails are breached.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula, assumptions, and validation checks clearly.
  - Decision: explain one actionable rule, one risk guardrail, and one fallback action.

## Required Extension Track (2+ Hours)
- Re-run today's notebook from a fresh kernel so outputs are reproducible without hidden state.
- Add one additional risk guardrail and verify how it changes trade/no-trade decisions.
- Document one failure mode, one mitigation, and one escalation rule for production use.

Extension completion checks:
- [ ] Notebook restarted and all coding cells rerun successfully
- [ ] At least one extra validation/edge-case test added
- [ ] Risk guardrail and fallback action documented

## Reflection Question
Which sleeve contributes most net alpha after execution costs?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Full notebook rerun completed from clean kernel
- [ ] Reflection logged in progress tracker


## Week 20 Day 05 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [14]:
# Advanced strategy demo: cross-sectional momentum spread on real assets
np.random.seed(2005)
assets = ['SPY', 'QQQ', 'IWM', 'EFA', 'EEM', 'TLT', 'GLD', 'HYG']
prices = load_market_prices(assets, start='2015-01-01')

mom_63 = prices.pct_change(63).iloc[-1]
next_5 = prices.pct_change().tail(5).mean()
signals = mom_63.dropna().sort_values(ascending=False)

n = max(2, len(signals) // 4)
long_bucket = list(signals.head(n).index)
short_bucket = list(signals.tail(n).index)
spread = float(next_5.loc[long_bucket].mean() - next_5.loc[short_bucket].mean())

{
    'long_bucket': long_bucket,
    'short_bucket': short_bucket,
    'spread_return_proxy': spread,
    'signal_snapshot': signals.round(4).to_dict(),
}


{'long_bucket': ['EEM', 'GLD'],
 'short_bucket': ['HYG', 'TLT'],
 'spread_return_proxy': 0.005495435804020488,
 'signal_snapshot': {'EEM': 0.0955,
  'GLD': 0.0534,
  'EFA': 0.051,
  'QQQ': 0.0449,
  'IWM': 0.0405,
  'SPY': 0.0287,
  'HYG': 0.0099,
  'TLT': -0.003}}

## ReAct Verification Cell
Validate trade logic with a risk guardrail before reading the model quiz answers.

In [15]:
# ReAct-style verification: observe -> reason -> act -> verify
np.random.seed(11005)

observe_tickers = ['SPY', 'QQQ', 'TLT']
observe_prices = load_market_prices(observe_tickers, start='2020-01-01')
observe_returns = observe_prices.pct_change().dropna()

if observe_returns.empty:
    raise ValueError('No returns available for verification checks')

ann_vol = float(observe_returns['SPY'].std() * np.sqrt(252))
ann_ret = float((1 + observe_returns['SPY']).prod() ** (252 / len(observe_returns)) - 1)
sharpe_proxy = float((ann_ret - 0.03) / max(ann_vol, 1e-8))

# Risk-first deployment gate used in realistic interview responses.
guardrail = 'de-risk' if ann_vol > 0.30 else 'monitor'
decision = 'deploy-paper-trade' if sharpe_proxy > 0.40 and guardrail == 'monitor' else 'hold-and-review'

verification = {
    'topic': 'Capstone assembly and governance',
    'week': 20,
    'day': 5,
    'observe_annual_return': ann_ret,
    'observe_annual_vol': ann_vol,
    'reason_sharpe_proxy': sharpe_proxy,
    'act_guardrail': guardrail,
    'verify_decision': decision,
}

verification


{'topic': 'Capstone assembly and governance',
 'week': 20,
 'day': 5,
 'observe_annual_return': 0.14923384092579295,
 'observe_annual_vol': 0.20501276581162983,
 'reason_sharpe_proxy': 0.5815922752603006,
 'act_guardrail': 'monitor',
 'verify_decision': 'deploy-paper-trade'}

## Week 20 Day 05 Quiz

Topic: **Capstone assembly and governance**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Explain Cross-Sectional Z-Score (z = (x - mu) / sigma) and define every symbol clearly.
2. Risk question: What hard guardrail would you enforce before deployment?
3. Data question: Which validation checks must pass before trusting this output?
4. Production question: Why does this topic matter in live systems, not just in notebooks?
5. Escalation question: If the metric degrades for three sessions, what is your fallback action?

Hint for Q2: throttle sizing when implementation shortfall worsens beyond policy.
Hint for Q3: capacity and borrow constraints before execution
Scoring: full credit requires notation correctness, one numeric example, explicit guardrail, and fallback path.

In [16]:
# Quiz model answers (auto-generated)
signal_x = 1.48
signal_mu = 0.92
signal_sigma = 0.28
metric_value = (signal_x - signal_mu) / signal_sigma
metric_name = 'cross_sectional_z'

print('Interview Question 1 (model answer):')
print('  Formula focus: Cross-Sectional Z-Score')
print('  Formula text : z = (x - mu) / sigma')
print('  Scenario     : crowded-factor unwind')
print('  Numeric value:', round(float(metric_value), 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce this guardrail before deployment:')
print('  throttle sizing when implementation shortfall exceeds threshold.')

print('\nInterview Question 3 (model answer):')
print('  I validate timestamps, missing data handling, and assumption consistency before taking risk.')

print('\nInterview Question 4 (model answer):')
print('  Topic:', 'Capstone assembly and governance')
print('  This matters because production systems need reproducible metrics, explicit controls,')
print('  and a fallback decision path when stress conditions invalidate baseline assumptions.')

print('\nInterview Question 5 (model answer):')
print('  Fallback action: halve sizing and tighten execution limits.')

print('\nNumeric verification:')
print('  metric_name  =', metric_name)
print('  metric_value =', round(float(metric_value), 6))


Interview Question 1 (model answer):
  Formula focus: Cross-Sectional Z-Score
  Formula text : z = (x - mu) / sigma
  Scenario     : crowded-factor unwind
  Numeric value: 2.0

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce this guardrail before deployment:
  throttle sizing when implementation shortfall exceeds threshold.

Interview Question 3 (model answer):
  I validate timestamps, missing data handling, and assumption consistency before taking risk.

Interview Question 4 (model answer):
  Topic: Capstone assembly and governance
  This matters because production systems need reproducible metrics, explicit controls,
  and a fallback decision path when stress conditions invalidate baseline assumptions.

Interview Question 5 (model answer):
  Fallback action: halve sizing and tighten execution limits.

Numeric verification:
  metric_name  = cross_sectional_z
  metric_value = 2.0


# Week 20 Day 06: Revision Sprint

## Study Duration
- Planned effort: 6-10 hours/day
- Required minimum: 6 hours across recall, rebuild, rerun, and retest blocks.

## 6-10 Hour Daily Structure
- **Core Block 1 (60 min):** Recall formulas, notation, and units without notes.
- **Core Block 2 (75 min):** Rebuild one weekday workflow from memory and compare with reference output.
- **Core Block 3 (60 min):** Rerun at least two notebooks from a clean kernel and capture mismatches.
- **Core Block 4 (60 min):** Diagnose root causes and update safeguards in code/comments.
- **Core Block 5 (45 min):** Interview drill, quiz, and escalation-rule rehearsal.
- **Required Extension Block A (60 min):** Re-test on one alternate ticker universe and one stress window.
- **Required Extension Block B (60 min):** Produce a remediation log with before/after evidence.
- **Optional Deep Work (0-4 hours):** Add robustness tests, monitor drift controls, and harden reporting.

## Why It Matters in Quant
Revision day converts fragile understanding into deployment-ready reliability by forcing recall, rebuild, and guardrail validation.

## Continuity and Handoff
- Previous checkpoint: Week 20 Day 05: Capstone assembly and governance
- Previous lesson file: content/week-20/day-05.md
- Today's deliverable: Revision checklist with corrected errors and next-week retest priorities.
- Next handoff target: Week 20 Day 07: Portfolio Project
- Next lesson file: content/week-20/day-07.md

## Revision Plan
- 90 minutes: active recall of weekday concepts and manual formula rewrite.
- 90 minutes: rebuild one weekday code workflow from memory.
- 90 minutes: restart kernel and rerun at least two day notebooks end-to-end.
- 90 minutes: error-log triage, retest plan, and guardrail refinement.
- Optional 0-4 hours: deepen weak areas with extra interview drill and code hardening.

## Focus Areas
- Re-check slippage assumptions against turnover
- Review blend risk-budget consistency
- Update execution governance checklist

## Mathematical Foundations (LaTeX)
### Formula 1: Cross-Sectional Z
$$
z_{i,t}=\frac{x_{i,t}-\mu_t}{\sigma_t}
$$
**Plain-English interpretation:** Universe-normalized signal.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Measure pre-cost and post-cost values to verify execution frictions do not erase signal edge.

### Formula 2: Information Coefficient
$$
IC_t=Corr(score_{i,t},r_{i,t+1})
$$
**Plain-English interpretation:** Signal/forward-return linkage.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Measure pre-cost and post-cost values to verify execution frictions do not erase signal edge.

### Formula 3: IC t-Statistic
$$
t_{IC}=\frac{\bar{IC}}{Std(IC)/\sqrt{T}}
$$
**Plain-English interpretation:** Signal persistence test.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Measure pre-cost and post-cost values to verify execution frictions do not erase signal edge.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $IC$ | Information coefficient | correlation | 0.04 |
| $ADV$ | Average daily volume | shares/day | 12M |
| $IS$ | Implementation shortfall | basis points | 14.2 bps |

## Real Trading Example
- Instruments: SPY, IWM, EFA, EEM
- Macro overlay (FRED): DFF, BAMLH0A0HYM2
- Suggested window: 2018-01-01 to 2026-03-31
- Stress windows to inspect: 2020-03 to 2020-05, 2023-03 to 2023-06
- Scenario context: crowded-factor unwind and liquidity stress
- Day objective: Rebuild two weekday workflows and compare decision outputs after a clean rerun.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull listed FRED series and join strictly by release-aware timestamps (no look-ahead).
3. Compute today's formulas and compare calm vs stress-window behavior.
4. Translate outputs into one explicit trade action and one hard risk guardrail.
5. Validate that the decision is consistent with topic 'Revision Sprint'.

## Step-by-Step Solved Problems
### Solved Problem 1: Factor z-score
Given:
- Signal=1.60, mean=0.70, std=0.45.
Solution:
1. $z=\frac{x-\mu}{\sigma}$.
2. z=(1.60-0.70)/0.45 = 2.00.
Final answer: Signal z-score = 2.00.
Common trap: Reporting gross signal performance without implementation costs or capacity constraints.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: IC t-stat
Given:
- Mean IC=0.045, std(IC)=0.018, T=12 months.
Solution:
1. $t=\frac{\bar{IC}}{s/\sqrt{T}}$.
2. t = 0.045/(0.018/sqrt(12)) = 8.66.
Final answer: IC t-stat = 8.66.
Common trap: Reporting gross signal performance without implementation costs or capacity constraints.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Implementation shortfall
Given:
- Arrival=101.20, execution=101.36.
Solution:
1. $IS_{bps}=10^4\frac{p_{exec}-p_{arr}}{p_{arr}}$.
2. IS_bps = 10000*(0.16/101.20) = 15.81.
Final answer: Implementation shortfall = 15.81 bps.
Common trap: Reporting gross signal performance without implementation costs or capacity constraints.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Rebuild one workflow from memory, rerun from clean kernel, and document one corrected failure mode.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
factor_scores = compute_factor_scores(universe_frame)
signals = build_long_short_buckets(factor_scores, q=0.2)
execution_cost = estimate_slippage(signals, adv_frame)
net_pnl = backtest_with_costs(signals, returns, execution_cost)
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive two key formulas from this week with units and constraints.
2. Rebuild one weekday notebook workflow from memory and verify output parity.
3. Replay one stress regime and document one changed decision rule.
4. Add one extra validation test that would have caught a prior mistake.
5. Record a 60-second explanation of one corrected failure mode.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 20 Day 06): Explain Cross-Sectional Z and define every symbol clearly for a crowded-factor unwind with liquidity deterioration.
   - Model answer: "I use Cross-Sectional Z as a decision bridge from market observations to position sizing. The formula is $z_{i,t}=\frac{x_{i,t}-\mu_t}{\sigma_t}$. I define each symbol with units first, then compute one concrete value, and finally state what trade action changes because of the result in this regime."
2. Risk manager question: Using one real ticker from this lesson, what hard guardrail would you enforce before live deployment?
   - Model answer: "I would run the workflow on SPY and a stress-sensitive peer, then throttle position sizing when estimated implementation shortfall rises above threshold. If the guardrail triggers, I switch to paper-trade monitoring and block new risk until diagnostics re-pass."
3. Data integrity question: Which checks must pass before you trust the output and place risk?
   - Model answer: "Before trading I verify survivorship-bias controls, borrow/shortability flags, and capacity-aware data completeness checks. If any check fails, I classify the run as non-tradable and log the incident."
4. Production question: Why does 'Revision Sprint' matter in live trading systems?
   - Model answer: "Revision Sprint matters because signal quality is meaningless unless net-of-cost and capacity-aware in live execution. In production I need reproducible calculations, explicit control limits, and escalation rules that survive stress windows."
5. Decision question: If your key metric degrades for three consecutive sessions, what is your fallback plan?
   - Model answer: "I halve position sizes, widen execution limits, and re-open capacity only after slippage improves. I only restore risk after rerun evidence confirms that assumptions are stable again."

Scoring rubric:
- 10/10: correct notation, one numeric example, explicit guardrail, data checks, and escalation path.
- 8/10: mostly correct notation plus a clear guardrail and fallback action.
- 6/10: partial correctness but vague controls or missing data validation.
- Below 6/10: formula recall without decision-quality risk controls.

### Interview Drill
- Prompt: "Walk me through Revision Sprint in an execution review with rising slippage and crowding risk."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision under constraints.
  3. Awareness of edge cases, costs, and failure modes.
  4. Clear escalation rule when guardrails are breached.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula, assumptions, and validation checks clearly.
  - Decision: explain one actionable rule, one risk guardrail, and one fallback action.

## Required Extension Track (2+ Hours)
- Re-run today's notebook from a fresh kernel so outputs are reproducible without hidden state.
- Add one additional risk guardrail and verify how it changes trade/no-trade decisions.
- Document one failure mode, one mitigation, and one escalation rule for production use.

Extension completion checks:
- [ ] Notebook restarted and all coding cells rerun successfully
- [ ] At least one extra validation/edge-case test added
- [ ] Risk guardrail and fallback action documented

## Reflection Question
Which failure mode still appears after reruns, and what concrete control will prevent it next week?

## Completion Checklist
- [ ] Updated error log entries
- [ ] Weak concepts re-tested
- [ ] Two notebooks rerun from fresh kernel
- [ ] Next-week risk list prepared
- [ ] Interview drill recorded with one fallback action


## Week 20 Day 06 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [17]:
# Revision sprint demo: rebuild weekly core diagnostics
np.random.seed(2006)
prices = load_market_prices(['SPY', 'QQQ', 'AAPL'], start='2018-01-01')
returns = prices.pct_change().dropna()

summary = pd.DataFrame({
    'annual_return': returns.mean() * 252,
    'annual_vol': returns.std() * np.sqrt(252),
    'max_drawdown': (prices / prices.cummax() - 1).min(),
})
summary['sharpe_proxy'] = (summary['annual_return'] - 0.03) / summary['annual_vol'].replace(0, np.nan)
summary = summary.sort_values('sharpe_proxy', ascending=False)

print('Revision diagnostics (best risk-adjusted first):')
summary.round(4)


Revision diagnostics (best risk-adjusted first):


,annual_return,annual_vol,max_drawdown,sharpe_proxy
Ticker,,,,
AAPL,0.277000,0.306000,-0.385200,0.807300
QQQ,0.205600,0.238800,-0.351200,0.735400
SPY,0.151700,0.193100,-0.337200,0.630400


## ReAct Verification Cell
Validate trade logic with a risk guardrail before reading the model quiz answers.

In [18]:
# ReAct-style verification: observe -> reason -> act -> verify
np.random.seed(11006)

observe_tickers = ['SPY', 'QQQ', 'TLT']
observe_prices = load_market_prices(observe_tickers, start='2020-01-01')
observe_returns = observe_prices.pct_change().dropna()

if observe_returns.empty:
    raise ValueError('No returns available for verification checks')

ann_vol = float(observe_returns['SPY'].std() * np.sqrt(252))
ann_ret = float((1 + observe_returns['SPY']).prod() ** (252 / len(observe_returns)) - 1)
sharpe_proxy = float((ann_ret - 0.03) / max(ann_vol, 1e-8))

# Risk-first deployment gate used in realistic interview responses.
guardrail = 'de-risk' if ann_vol > 0.30 else 'monitor'
decision = 'deploy-paper-trade' if sharpe_proxy > 0.40 and guardrail == 'monitor' else 'hold-and-review'

verification = {
    'topic': 'Revision Sprint',
    'week': 20,
    'day': 6,
    'observe_annual_return': ann_ret,
    'observe_annual_vol': ann_vol,
    'reason_sharpe_proxy': sharpe_proxy,
    'act_guardrail': guardrail,
    'verify_decision': decision,
}

verification


{'topic': 'Revision Sprint',
 'week': 20,
 'day': 6,
 'observe_annual_return': 0.14923384092579295,
 'observe_annual_vol': 0.20501276581162983,
 'reason_sharpe_proxy': 0.5815922752603006,
 'act_guardrail': 'monitor',
 'verify_decision': 'deploy-paper-trade'}

## Week 20 Day 06 Quiz

Topic: **Revision Sprint**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Explain Cross-Sectional Z-Score (z = (x - mu) / sigma) and define every symbol clearly.
2. Risk question: What hard guardrail would you enforce before deployment?
3. Data question: Which validation checks must pass before trusting this output?
4. Production question: Why does this topic matter in live systems, not just in notebooks?
5. Escalation question: If the metric degrades for three sessions, what is your fallback action?

Hint for Q2: throttle sizing when implementation shortfall worsens beyond policy.
Hint for Q3: capacity and borrow constraints before execution
Scoring: full credit requires notation correctness, one numeric example, explicit guardrail, and fallback path.

In [19]:
# Quiz model answers (auto-generated)
signal_x = 1.48
signal_mu = 0.92
signal_sigma = 0.28
metric_value = (signal_x - signal_mu) / signal_sigma
metric_name = 'cross_sectional_z'

print('Interview Question 1 (model answer):')
print('  Formula focus: Cross-Sectional Z-Score')
print('  Formula text : z = (x - mu) / sigma')
print('  Scenario     : crowded-factor unwind')
print('  Numeric value:', round(float(metric_value), 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce this guardrail before deployment:')
print('  throttle sizing when implementation shortfall exceeds threshold.')

print('\nInterview Question 3 (model answer):')
print('  I validate timestamps, missing data handling, and assumption consistency before taking risk.')

print('\nInterview Question 4 (model answer):')
print('  Topic:', 'Revision Sprint')
print('  This matters because production systems need reproducible metrics, explicit controls,')
print('  and a fallback decision path when stress conditions invalidate baseline assumptions.')

print('\nInterview Question 5 (model answer):')
print('  Fallback action: halve sizing and tighten execution limits.')

print('\nNumeric verification:')
print('  metric_name  =', metric_name)
print('  metric_value =', round(float(metric_value), 6))


Interview Question 1 (model answer):
  Formula focus: Cross-Sectional Z-Score
  Formula text : z = (x - mu) / sigma
  Scenario     : crowded-factor unwind
  Numeric value: 2.0

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce this guardrail before deployment:
  throttle sizing when implementation shortfall exceeds threshold.

Interview Question 3 (model answer):
  I validate timestamps, missing data handling, and assumption consistency before taking risk.

Interview Question 4 (model answer):
  Topic: Revision Sprint
  This matters because production systems need reproducible metrics, explicit controls,
  and a fallback decision path when stress conditions invalidate baseline assumptions.

Interview Question 5 (model answer):
  Fallback action: halve sizing and tighten execution limits.

Numeric verification:
  metric_name  = cross_sectional_z
  metric_value = 2.0


# Week 20 Day 07: Portfolio Project

## Study Duration
- Planned effort: 6-10 hours/day
- Required minimum: 6 hours for implementation, validation, and communication drills.

## 6-10 Hour Daily Structure
- **Core Block 1 (60 min):** Restate objective, assumptions, and measurable success criteria.
- **Core Block 2 (75 min):** Build/clean data pipeline and verify timestamp integrity.
- **Core Block 3 (75 min):** Implement project logic and validate formulas against spot checks.
- **Core Block 4 (60 min):** Produce diagnostics, stress checks, and fallback pathways.
- **Core Block 5 (45 min):** Deliver interview-style defense with risk controls and escalation.
- **Required Extension Block A (60 min):** Re-run project on alternate assumptions and compare drift.
- **Required Extension Block B (60 min):** Prepare production memo and launch/no-launch decision log.
- **Optional Deep Work (0-4 hours):** Expand tests, improve monitoring, and polish stakeholder narrative.

## Why It Matters in Quant
Project day is where research quality meets execution discipline and communication quality under risk constraints.

## Continuity and Handoff
- Previous checkpoint: Week 20 Day 06: Revision Sprint
- Previous lesson file: content/week-20/day-06.md
- Today's deliverable: Capstone 5: Multi-strategy paper portfolio
- Next handoff target: Week 21 Day 01: Program targeting and fit matrix
- Next lesson file: content/week-21/day-01.md

## Project Blueprint
### Project Title
Capstone 5: Multi-strategy paper portfolio

### Problem Statement
Construct a multi-sleeve paper portfolio with microstructure-aware execution assumptions.

### Data Sources
- Strategy sleeve signals
- Liquidity proxies
- Cost model inputs

### Implementation Steps
1. Design sleeve blend
2. Apply execution-cost assumptions
3. Backtest net performance
4. Analyze attribution
5. Produce governance report

### Evaluation Metrics
- Net Sharpe proxy
- Cost drag
- Capacity proxy
- Risk-budget adherence

### Execution Standard
- [ ] Notebook/script runs from clean start without hidden state
- [ ] Outputs include at least one diagnostic table and one chart
- [ ] One explicit risk guardrail and fallback action are documented

### Deliverables
- Notebook or script output
- One-page summary memo
- Tracker update with completion and lessons learned

## Mathematical Foundations (LaTeX)
### Formula 1: Information Coefficient
$$
IC_t=Corr(score_{i,t},r_{i,t+1})
$$
**Plain-English interpretation:** Signal/forward-return linkage.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Measure pre-cost and post-cost values to verify execution frictions do not erase signal edge.

### Formula 2: IC t-Statistic
$$
t_{IC}=\frac{\bar{IC}}{Std(IC)/\sqrt{T}}
$$
**Plain-English interpretation:** Signal persistence test.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Measure pre-cost and post-cost values to verify execution frictions do not erase signal edge.

### Formula 3: Spread Z-Score
$$
z_t=\frac{s_t-\mu_s}{\sigma_s}
$$
**Plain-English interpretation:** Stat-arb entry normalization.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Measure pre-cost and post-cost values to verify execution frictions do not erase signal edge.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $IC$ | Information coefficient | correlation | 0.04 |
| $ADV$ | Average daily volume | shares/day | 12M |
| $IS$ | Implementation shortfall | basis points | 14.2 bps |

## Real Trading Example
- Instruments: SPY, IWM, EFA, EEM
- Macro overlay (FRED): DFF, BAMLH0A0HYM2
- Suggested window: 2018-01-01 to 2026-03-31
- Stress windows to inspect: 2020-03 to 2020-05, 2023-03 to 2023-06
- Scenario context: crowded-factor unwind and liquidity stress
- Day objective: Deliver a capstone-quality notebook and summarize one trade-off under stress assumptions.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull listed FRED series and join strictly by release-aware timestamps (no look-ahead).
3. Compute today's formulas and compare calm vs stress-window behavior.
4. Translate outputs into one explicit trade action and one hard risk guardrail.
5. Validate that the decision is consistent with topic 'Portfolio Project'.

## Step-by-Step Solved Problems
### Solved Problem 1: Factor z-score
Given:
- Signal=1.60, mean=0.70, std=0.45.
Solution:
1. $z=\frac{x-\mu}{\sigma}$.
2. z=(1.60-0.70)/0.45 = 2.00.
Final answer: Signal z-score = 2.00.
Common trap: Reporting gross signal performance without implementation costs or capacity constraints.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: IC t-stat
Given:
- Mean IC=0.045, std(IC)=0.018, T=12 months.
Solution:
1. $t=\frac{\bar{IC}}{s/\sqrt{T}}$.
2. t = 0.045/(0.018/sqrt(12)) = 8.66.
Final answer: IC t-stat = 8.66.
Common trap: Reporting gross signal performance without implementation costs or capacity constraints.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Implementation shortfall
Given:
- Arrival=101.20, execution=101.36.
Solution:
1. $IS_{bps}=10^4\frac{p_{exec}-p_{arr}}{p_{arr}}$.
2. IS_bps = 10000*(0.16/101.20) = 15.81.
Final answer: Implementation shortfall = 15.81 bps.
Common trap: Reporting gross signal performance without implementation costs or capacity constraints.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Ship the project notebook with reproducible outputs, controls, and one escalation rule.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
factor_scores = compute_factor_scores(universe_frame)
signals = build_long_short_buckets(factor_scores, q=0.2)
execution_cost = estimate_slippage(signals, adv_frame)
net_pnl = backtest_with_costs(signals, returns, execution_cost)
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. State project objective and one hard failure condition in under 45 seconds.
2. Validate one formula output against a manual spot-check.
3. Show one stress scenario where your decision changes and explain why.
4. Add one edge-case test and one fallback rule to the notebook.
5. Deliver a one-minute PM summary with one risk guardrail.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 20 Day 07): Explain Information Coefficient and define every symbol clearly for a crowded-factor unwind with liquidity deterioration.
   - Model answer: "I use Information Coefficient as a decision bridge from market observations to position sizing. The formula is $IC_t=Corr(score_{i,t},r_{i,t+1})$. I define each symbol with units first, then compute one concrete value, and finally state what trade action changes because of the result in this regime."
2. Risk manager question: Using one real ticker from this lesson, what hard guardrail would you enforce before live deployment?
   - Model answer: "I would run the workflow on SPY and a stress-sensitive peer, then throttle position sizing when estimated implementation shortfall rises above threshold. If the guardrail triggers, I switch to paper-trade monitoring and block new risk until diagnostics re-pass."
3. Data integrity question: Which checks must pass before you trust the output and place risk?
   - Model answer: "Before trading I verify survivorship-bias controls, borrow/shortability flags, and capacity-aware data completeness checks. If any check fails, I classify the run as non-tradable and log the incident."
4. Production question: Why does 'Portfolio Project' matter in live trading systems?
   - Model answer: "Portfolio Project matters because signal quality is meaningless unless net-of-cost and capacity-aware in live execution. In production I need reproducible calculations, explicit control limits, and escalation rules that survive stress windows."
5. Decision question: If your key metric degrades for three consecutive sessions, what is your fallback plan?
   - Model answer: "I halve position sizes, widen execution limits, and re-open capacity only after slippage improves. I only restore risk after rerun evidence confirms that assumptions are stable again."

Scoring rubric:
- 10/10: correct notation, one numeric example, explicit guardrail, data checks, and escalation path.
- 8/10: mostly correct notation plus a clear guardrail and fallback action.
- 6/10: partial correctness but vague controls or missing data validation.
- Below 6/10: formula recall without decision-quality risk controls.

### Interview Drill
- Prompt: "Walk me through Portfolio Project in an execution review with rising slippage and crowding risk."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision under constraints.
  3. Awareness of edge cases, costs, and failure modes.
  4. Clear escalation rule when guardrails are breached.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula, assumptions, and validation checks clearly.
  - Decision: explain one actionable rule, one risk guardrail, and one fallback action.

## Required Extension Track (2+ Hours)
- Re-run today's notebook from a fresh kernel so outputs are reproducible without hidden state.
- Add one additional risk guardrail and verify how it changes trade/no-trade decisions.
- Document one failure mode, one mitigation, and one escalation rule for production use.

Extension completion checks:
- [ ] Notebook restarted and all coding cells rerun successfully
- [ ] At least one extra validation/edge-case test added
- [ ] Risk guardrail and fallback action documented

## Reflection Question
What single risk control would block launch today, and what evidence would clear it?

## Completion Checklist
- [ ] Project notebook runs cleanly from fresh kernel
- [ ] Risk guardrail and fallback action documented
- [ ] Stress scenario comparison completed
- [ ] One-page summary memo finalized
- [ ] Launch/no-launch decision recorded with evidence


## Week 20 Day 07 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [20]:
# Project day demo: mini portfolio report with trade recommendation
np.random.seed(2007)
assets = ['SPY', 'QQQ', 'TLT', 'GLD']
prices = load_market_prices(assets, start='2019-01-01')
returns = prices.pct_change().dropna()

annual_return = returns.mean() * 252
annual_vol = returns.std() * np.sqrt(252)
score = (annual_return - 0.03) / annual_vol.replace(0, np.nan)

report = pd.DataFrame({
    'annual_return': annual_return,
    'annual_vol': annual_vol,
    'sharpe_proxy': score,
}).sort_values('sharpe_proxy', ascending=False)

top_asset = report.index[0]
print('Project summary:')
print(report.round(4))
print(f"\nSuggested focus asset for follow-up research: {top_asset}")


Project summary:
        annual_return  annual_vol  sharpe_proxy
Ticker                                         
GLD          0.194100    0.172700      0.950000
QQQ          0.232200    0.240200      0.841900
SPY          0.177800    0.196000      0.754000
TLT         -0.005100    0.162600     -0.215600

Suggested focus asset for follow-up research: GLD


## ReAct Verification Cell
Validate trade logic with a risk guardrail before reading the model quiz answers.

In [21]:
# ReAct-style verification: observe -> reason -> act -> verify
np.random.seed(11007)

observe_tickers = ['SPY', 'QQQ', 'TLT']
observe_prices = load_market_prices(observe_tickers, start='2020-01-01')
observe_returns = observe_prices.pct_change().dropna()

if observe_returns.empty:
    raise ValueError('No returns available for verification checks')

ann_vol = float(observe_returns['SPY'].std() * np.sqrt(252))
ann_ret = float((1 + observe_returns['SPY']).prod() ** (252 / len(observe_returns)) - 1)
sharpe_proxy = float((ann_ret - 0.03) / max(ann_vol, 1e-8))

# Risk-first deployment gate used in realistic interview responses.
guardrail = 'de-risk' if ann_vol > 0.30 else 'monitor'
decision = 'deploy-paper-trade' if sharpe_proxy > 0.40 and guardrail == 'monitor' else 'hold-and-review'

verification = {
    'topic': 'Portfolio Project',
    'week': 20,
    'day': 7,
    'observe_annual_return': ann_ret,
    'observe_annual_vol': ann_vol,
    'reason_sharpe_proxy': sharpe_proxy,
    'act_guardrail': guardrail,
    'verify_decision': decision,
}

verification


{'topic': 'Portfolio Project',
 'week': 20,
 'day': 7,
 'observe_annual_return': 0.14923384092579295,
 'observe_annual_vol': 0.20501276581162983,
 'reason_sharpe_proxy': 0.5815922752603006,
 'act_guardrail': 'monitor',
 'verify_decision': 'deploy-paper-trade'}

## Week 20 Day 07 Quiz

Topic: **Portfolio Project**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Explain Cross-Sectional Z-Score (z = (x - mu) / sigma) and define every symbol clearly.
2. Risk question: What hard guardrail would you enforce before deployment?
3. Data question: Which validation checks must pass before trusting this output?
4. Production question: Why does this topic matter in live systems, not just in notebooks?
5. Escalation question: If the metric degrades for three sessions, what is your fallback action?

Hint for Q2: throttle sizing when implementation shortfall worsens beyond policy.
Hint for Q3: capacity and borrow constraints before execution
Scoring: full credit requires notation correctness, one numeric example, explicit guardrail, and fallback path.

In [22]:
# Quiz model answers (auto-generated)
signal_x = 1.48
signal_mu = 0.92
signal_sigma = 0.28
metric_value = (signal_x - signal_mu) / signal_sigma
metric_name = 'cross_sectional_z'

print('Interview Question 1 (model answer):')
print('  Formula focus: Cross-Sectional Z-Score')
print('  Formula text : z = (x - mu) / sigma')
print('  Scenario     : crowded-factor unwind')
print('  Numeric value:', round(float(metric_value), 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce this guardrail before deployment:')
print('  throttle sizing when implementation shortfall exceeds threshold.')

print('\nInterview Question 3 (model answer):')
print('  I validate timestamps, missing data handling, and assumption consistency before taking risk.')

print('\nInterview Question 4 (model answer):')
print('  Topic:', 'Portfolio Project')
print('  This matters because production systems need reproducible metrics, explicit controls,')
print('  and a fallback decision path when stress conditions invalidate baseline assumptions.')

print('\nInterview Question 5 (model answer):')
print('  Fallback action: halve sizing and tighten execution limits.')

print('\nNumeric verification:')
print('  metric_name  =', metric_name)
print('  metric_value =', round(float(metric_value), 6))


Interview Question 1 (model answer):
  Formula focus: Cross-Sectional Z-Score
  Formula text : z = (x - mu) / sigma
  Scenario     : crowded-factor unwind
  Numeric value: 2.0

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce this guardrail before deployment:
  throttle sizing when implementation shortfall exceeds threshold.

Interview Question 3 (model answer):
  I validate timestamps, missing data handling, and assumption consistency before taking risk.

Interview Question 4 (model answer):
  Topic: Portfolio Project
  This matters because production systems need reproducible metrics, explicit controls,
  and a fallback decision path when stress conditions invalidate baseline assumptions.

Interview Question 5 (model answer):
  Fallback action: halve sizing and tighten execution limits.

Numeric verification:
  metric_name  = cross_sectional_z
  metric_value = 2.0
